# Multi-Agent Trading System - Backtesting Analysis

This notebook provides tools for backtesting the trading system and analyzing performance.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.database import DatabaseHandler
from data.models import TradeStatus

# Initialize database
db = DatabaseHandler()

## 1. Load Trade History

In [ ]:
# Get all closed trades
trades = db.get_trades(status=TradeStatus.CLOSED)

print(f"Total trades: {len(trades)}")

# Convert to DataFrame
trades_data = []
for trade in trades:
    trades_data.append({
        'symbol': trade.symbol,
        'direction': trade.direction.value,
        'entry_price': trade.entry_price,
        'exit_price': trade.exit_price,
        'profit_loss': trade.profit_loss,
        'confidence': trade.confidence,
        'entry_time': trade.entry_time,
        'exit_time': trade.exit_time,
        'is_winner': trade.is_winner
    })

df = pd.DataFrame(trades_data)
df.head()

## 2. Calculate Performance Metrics

In [ ]:
# Basic metrics
total_trades = len(df)
winning_trades = df['is_winner'].sum()
losing_trades = total_trades - winning_trades
win_rate = winning_trades / total_trades if total_trades > 0 else 0

# P&L metrics
total_pnl = df['profit_loss'].sum()
avg_win = df[df['is_winner']]['profit_loss'].mean() if winning_trades > 0 else 0
avg_loss = df[~df['is_winner']]['profit_loss'].mean() if losing_trades > 0 else 0

# Profit factor
total_wins = df[df['is_winner']]['profit_loss'].sum()
total_losses = abs(df[~df['is_winner']]['profit_loss'].sum())
profit_factor = total_wins / total_losses if total_losses > 0 else 0

print(f"""Performance Metrics:
Total Trades: {total_trades}
Win Rate: {win_rate:.1%}
Total P&L: ${total_pnl:.2f}
Avg Win: ${avg_win:.2f}
Avg Loss: ${avg_loss:.2f}
Profit Factor: {profit_factor:.2f}
""")

## 3. Equity Curve

In [ ]:
# Calculate cumulative P&L
df_sorted = df.sort_values('exit_time')
df_sorted['cumulative_pnl'] = df_sorted['profit_loss'].cumsum()

# Plot equity curve
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_sorted['exit_time'],
    y=df_sorted['cumulative_pnl'],
    mode='lines',
    name='Equity',
    line=dict(color='blue', width=2),
    fill='tozeroy'
))

fig.update_layout(
    title='Equity Curve',
    xaxis_title='Date',
    yaxis_title='Cumulative P&L ($)',
    height=500
)

fig.show()

## 4. Drawdown Analysis

In [ ]:
# Calculate drawdown
df_sorted['peak'] = df_sorted['cumulative_pnl'].cummax()
df_sorted['drawdown'] = df_sorted['peak'] - df_sorted['cumulative_pnl']
df_sorted['drawdown_pct'] = (df_sorted['drawdown'] / df_sorted['peak'].replace(0, 1)) * 100

max_drawdown = df_sorted['drawdown'].max()
max_drawdown_pct = df_sorted['drawdown_pct'].max()

print(f"Max Drawdown: ${max_drawdown:.2f} ({max_drawdown_pct:.1f}%)")

# Plot drawdown
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_sorted['exit_time'],
    y=df_sorted['drawdown'],
    mode='lines',
    name='Drawdown',
    line=dict(color='red', width=2),
    fill='tozeroy'
))

fig.update_layout(
    title='Drawdown Over Time',
    xaxis_title='Date',
    yaxis_title='Drawdown ($)',
    height=400
)

fig.show()

## 5. Performance by Symbol

In [ ]:
# Group by symbol
symbol_performance = df.groupby('symbol').agg({
    'profit_loss': ['count', 'sum', 'mean'],
    'is_winner': 'sum'
}).round(2)

symbol_performance.columns = ['Trades', 'Total P&L', 'Avg P&L', 'Wins']
symbol_performance['Win Rate'] = (symbol_performance['Wins'] / symbol_performance['Trades'] * 100).round(1)

print("Performance by Symbol:")
print(symbol_performance.sort_values('Total P&L', ascending=False))

## 6. Monthly Performance

In [ ]:
# Extract month
df_sorted['month'] = pd.to_datetime(df_sorted['exit_time']).dt.to_period('M')

# Monthly performance
monthly_pnl = df_sorted.groupby('month')['profit_loss'].sum()

# Plot monthly P&L
fig = go.Figure()

colors = ['green' if x > 0 else 'red' for x in monthly_pnl.values]

fig.add_trace(go.Bar(
    x=[str(x) for x in monthly_pnl.index],
    y=monthly_pnl.values,
    marker_color=colors
))

fig.update_layout(
    title='Monthly P&L',
    xaxis_title='Month',
    yaxis_title='P&L ($)',
    height=400
)

fig.show()

## 7. Trade Duration Analysis

In [ ]:
# Calculate trade duration
df['duration_hours'] = (pd.to_datetime(df['exit_time']) - pd.to_datetime(df['entry_time'])).dt.total_seconds() / 3600

# Duration statistics
print(f"""Trade Duration:
Average: {df['duration_hours'].mean():.1f} hours
Median: {df['duration_hours'].median():.1f} hours
Min: {df['duration_hours'].min():.1f} hours
Max: {df['duration_hours'].max():.1f} hours
""")

# Plot duration distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['duration_hours'],
    nbinsx=30,
    name='Trade Duration'
))

fig.update_layout(
    title='Trade Duration Distribution',
    xaxis_title='Duration (hours)',
    yaxis_title='Frequency',
    height=400
)

fig.show()

## 8. Confidence vs Performance

In [ ]:
# Group by confidence bins
df['confidence_bin'] = pd.cut(df['confidence'], bins=[0, 0.6, 0.7, 0.8, 0.9, 1.0])

confidence_performance = df.groupby('confidence_bin').agg({
    'profit_loss': ['count', 'sum', 'mean'],
    'is_winner': 'sum'
}).round(2)

confidence_performance.columns = ['Trades', 'Total P&L', 'Avg P&L', 'Wins']
confidence_performance['Win Rate'] = (confidence_performance['Wins'] / confidence_performance['Trades'] * 100).round(1)

print("Performance by Confidence Level:")
print(confidence_performance)

## 9. Export Report

In [ ]:
# Export comprehensive report to CSV
df_sorted.to_csv('../backtest_results.csv', index=False)
print("Report exported to backtest_results.csv")